# 第9回: HSARNN - 階層型 Spatial Attention RNN

このノートブックでは、**SARNN（Spatial Attention RNN）** の発展形である
**HSARNN（Hierarchical Spatial Attention RNN）** を学びます。
SARNNでは画像特徴量と関節角度を単一のRNNで処理していましたが、
HSARNNでは **階層構造** を導入し、関節角度とキーポイント（注意点）を
**分離して処理** した上で統合することで、より柔軟な表現学習を実現します。

## 1. 概要

### SARNNの限界

SARNNでは、画像からSpatial Softmaxで抽出したキーポイント座標と関節角度を
結合（concatenate）して単一のLSTMCellに入力していました。
この設計には以下の課題があります:

- **異なるモダリティの混在**: 関節角度（固有受容感覚）とキーポイント座標（視覚由来）は
  本質的に異なる情報であり、単一のRNNで同時に処理すると互いに干渉する可能性がある
- **スケーラビリティ**: センサ数やキーポイント数を増やすと、単一RNNの負荷が増大
- **解釈性**: どの情報がどのように処理されているか分離が困難

### 階層型アプローチ

HSARNNでは、入力を **2つの専用RNN** で別々に処理し、
**統合RNN（Union RNN）** で情報を交換します:

```
関節角度 xv ─→ [io_rnn1 (LSTM)] ←── union からの h1
                       ↓ h_rnn1
                       ├────────────→ [union_rnn (LSTM)] ←── h_rnn2
                       ↓                      ↓
                  関節デコーダ          統合隠れ状態 → h1, h2 に分配
                       ↓                      ↑
キーポイント xpt ─→ [io_rnn2 (LSTM)] ←── union からの h2
                       ↓ h_rnn2
                  ポイントデコーダ
```

**利点:**
- 各モダリティに特化したRNNで効率的に処理
- Union RNN がモダリティ間の依存関係を学習
- 各RNNの隠れ状態を独立に分析可能

## 2. セットアップ

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")

torch.manual_seed(42)
np.random.seed(42)

## 3. HierachicalRNNCell

HSARNNの中核となる階層型RNNセルを実装します。
このセルは以下の3つのLSTMCellから構成されます:

| コンポーネント | 入力 | 役割 |
|---|---|---|
| `io_rnn1` | 関節角度 xv | 関節角度の時系列処理 |
| `io_rnn2` | キーポイント xpt | キーポイントの時系列処理 |
| `union_rnn` | h_rnn1 と h_rnn2 の結合 | モダリティ間の情報統合 |

Union RNN の出力は線形層で分割され、各 io_rnn の隠れ状態にフィードバックされます。

In [ ]:
class HierachicalRNNCell(nn.Module):
    def __init__(self, input_dim1, input_dim2, rnn_dim=50, union_dim=20):
        super().__init__()
        self.rnn_dim = rnn_dim
        self.union_dim = union_dim
        # 関節角度用RNN
        self.io_rnn1 = nn.LSTMCell(input_dim1, rnn_dim)
        # キーポイント用RNN
        self.io_rnn2 = nn.LSTMCell(input_dim2, rnn_dim)
        # 統合RNN（両方の隠れ状態を入力）
        self.union_rnn = nn.LSTMCell(rnn_dim * 2, union_dim)
        # 統合RNNの出力を2つのRNNにフィードバック
        self.union_out = nn.Linear(union_dim, rnn_dim * 2)

    def get_initial_states(self, x):
        B, dev = x.shape[0], x.device
        z = lambda d: torch.zeros(B, d, device=dev)
        return (
            [z(self.rnn_dim), z(self.rnn_dim)],    # io_rnn1: (h, c)
            [z(self.rnn_dim), z(self.rnn_dim)],    # io_rnn2: (h, c)
            [z(self.union_dim), z(self.union_dim)], # union:   (h, c)
        )

    def forward(self, xv, xpt, states=None):
        if states is None:
            states = self.get_initial_states(xv)
        prev_rnn1, prev_rnn2, prev_union = states

        # 1. Union RNN: 前時刻の両RNNの隠れ状態を結合して入力
        inputs_u = torch.cat((prev_rnn1[0], prev_rnn2[0]), dim=-1)
        new_union = self.union_rnn(inputs_u, prev_union)

        # 2. Union出力を分割して各RNNにフィードバック
        _uh = self.union_out(new_union[0])
        r1s, r2s = torch.split(_uh, self.rnn_dim, dim=-1)

        # 3. 各io_rnn: union からの h とprev の c を使って更新
        new_rnn1 = self.io_rnn1(xv, [r1s, prev_rnn1[1]])
        new_rnn2 = self.io_rnn2(xpt, [r2s, prev_rnn2[1]])

        return new_rnn1, new_rnn2, new_union

### 動作確認

小さなダミーデータで HierachicalRNNCell が正しく動作するか確認します。

In [ ]:
# ダミーデータで動作確認
batch_size = 4
joint_dim = 7   # 関節角度の次元
point_dim = 10  # キーポイントの次元（5点 × 2座標）

cell = HierachicalRNNCell(input_dim1=joint_dim, input_dim2=point_dim, rnn_dim=50, union_dim=20).to(device)

xv = torch.randn(batch_size, joint_dim, device=device)
xpt = torch.randn(batch_size, point_dim, device=device)

# 1ステップ目
rnn1, rnn2, union = cell(xv, xpt)
print(f"io_rnn1 hidden: {rnn1[0].shape}")  # (B, rnn_dim)
print(f"io_rnn2 hidden: {rnn2[0].shape}")  # (B, rnn_dim)
print(f"union hidden:   {union[0].shape}")  # (B, union_dim)

# 2ステップ目（状態を引き継ぐ）
rnn1, rnn2, union = cell(xv, xpt, states=(rnn1, rnn2, union))
print(f"\n2ステップ目も正常に動作: rnn1 h={rnn1[0].shape}")

## 4. SpatialSoftmax と InverseSpatialSoftmax

SARNNと同様に、画像からキーポイントを抽出するための **SpatialSoftmax** と、
キーポイントから注意マップを再構成する **InverseSpatialSoftmax** を実装します。

In [ ]:
class SpatialSoftmax(nn.Module):
    """特徴マップからキーポイント座標を抽出する。"""
    def __init__(self, temperature=1.0):
        super().__init__()
        self.temperature = temperature

    def forward(self, x):
        # x: (B, C, H, W)
        B, C, H, W = x.shape

        # 正規化座標のグリッドを作成
        pos_y = torch.linspace(-1.0, 1.0, H, device=x.device)
        pos_x = torch.linspace(-1.0, 1.0, W, device=x.device)

        # Softmax で確率分布に変換
        flat = x.view(B, C, -1) / self.temperature
        softmax = F.softmax(flat, dim=-1)
        softmax = softmax.view(B, C, H, W)

        # 期待値としてキーポイント座標を計算
        ey = (softmax.sum(dim=-1) * pos_y.view(1, 1, -1)).sum(dim=-1)
        ex = (softmax.sum(dim=-2) * pos_x.view(1, 1, -1)).sum(dim=-1)

        # (B, C*2): 各チャネルの (y, x) 座標を結合
        return torch.stack([ey, ex], dim=-1).view(B, -1)


class InverseSpatialSoftmax(nn.Module):
    """キーポイント座標からガウシアン注意マップを再構成する。"""
    def __init__(self, height, width, n_keypoints):
        super().__init__()
        self.height = height
        self.width = width
        self.n_keypoints = n_keypoints

        pos_y = torch.linspace(-1.0, 1.0, height)
        pos_x = torch.linspace(-1.0, 1.0, width)
        grid_y, grid_x = torch.meshgrid(pos_y, pos_x, indexing="ij")
        self.register_buffer("grid_y", grid_y)
        self.register_buffer("grid_x", grid_x)

    def forward(self, points, sigma=0.1):
        # points: (B, n_keypoints*2)
        B = points.shape[0]
        pts = points.view(B, self.n_keypoints, 2)
        py = pts[:, :, 0].unsqueeze(-1).unsqueeze(-1)
        px = pts[:, :, 1].unsqueeze(-1).unsqueeze(-1)

        gy = self.grid_y.unsqueeze(0).unsqueeze(0)
        gx = self.grid_x.unsqueeze(0).unsqueeze(0)

        heatmap = torch.exp(-((gy - py)**2 + (gx - px)**2) / (2 * sigma**2))
        return heatmap  # (B, n_keypoints, H, W)

## 5. HSARNNモデル

HSARNNは以下の構成です:

1. **画像エンコーダ（CNN）**: 画像を特徴マップに変換
2. **SpatialSoftmax**: 特徴マップからキーポイント座標を抽出
3. **HierachicalRNNCell**: 関節角度とキーポイントを階層的に処理
4. **関節デコーダ**: io_rnn1 の隠れ状態から関節角度を予測
5. **ポイントデコーダ**: io_rnn2 の隠れ状態からキーポイント座標を予測
6. **画像デコーダ（CNN）**: キーポイントの注意マップを元に画像を再構成

In [ ]:
class HSARNN(nn.Module):
    def __init__(self, image_shape=(3, 64, 64), joint_dim=7,
                 n_keypoints=5, rnn_dim=50, union_dim=20):
        super().__init__()
        C, H, W = image_shape
        self.n_keypoints = n_keypoints
        self.joint_dim = joint_dim
        self.image_shape = image_shape

        # --- 画像エンコーダ ---
        self.encoder = nn.Sequential(
            nn.Conv2d(C, 16, 3, stride=2, padding=1),  # -> 32x32
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), # -> 16x16
            nn.ReLU(),
            nn.Conv2d(32, n_keypoints, 3, padding=1),  # -> 16x16, n_keypoints ch
        )

        # --- SpatialSoftmax ---
        self.spatial_softmax = SpatialSoftmax()

        # --- InverseSpatialSoftmax ---
        self.feat_h, self.feat_w = H // 4, W // 4  # エンコーダ後のサイズ
        self.inv_spatial_softmax = InverseSpatialSoftmax(
            self.feat_h, self.feat_w, n_keypoints
        )

        # --- 階層型RNNセル ---
        point_dim = n_keypoints * 2
        self.rnn_cell = HierachicalRNNCell(
            input_dim1=joint_dim,
            input_dim2=point_dim,
            rnn_dim=rnn_dim,
            union_dim=union_dim,
        )

        # --- デコーダ ---
        self.joint_decoder = nn.Linear(rnn_dim, joint_dim)
        self.point_decoder = nn.Linear(rnn_dim, point_dim)

        # --- 画像デコーダ ---
        self.image_decoder = nn.Sequential(
            nn.Conv2d(n_keypoints, 32, 3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2),              # -> 32x32
            nn.Conv2d(32, 16, 3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2),              # -> 64x64
            nn.Conv2d(16, C, 3, padding=1),
            nn.Sigmoid(),
        )

    def encode_image(self, img):
        """画像 -> キーポイント座標"""
        feat = self.encoder(img)
        pts = self.spatial_softmax(feat)
        return pts, feat

    def decode_image(self, pts):
        """キーポイント座標 -> 再構成画像"""
        heatmap = self.inv_spatial_softmax(pts)
        return self.image_decoder(heatmap)

    def forward(self, images, joints, seq_len=None):
        """
        Args:
            images: (B, T, C, H, W) 画像系列
            joints: (B, T, joint_dim) 関節角度系列
        Returns:
            dict with pred_images, pred_joints, pred_points, enc_points
        """
        B, T = images.shape[:2]
        states = None

        pred_images, pred_joints, pred_points, enc_points = [], [], [], []

        for t in range(T):
            img_t = images[:, t]
            jnt_t = joints[:, t]

            # 画像エンコード -> キーポイント
            pts_t, _ = self.encode_image(img_t)
            enc_points.append(pts_t)

            # 階層型RNN
            rnn1, rnn2, union = self.rnn_cell(jnt_t, pts_t, states)
            states = (rnn1, rnn2, union)

            # デコード
            pred_jnt = self.joint_decoder(rnn1[0])
            pred_pt = self.point_decoder(rnn2[0])
            pred_img = self.decode_image(pred_pt)

            pred_joints.append(pred_jnt)
            pred_points.append(pred_pt)
            pred_images.append(pred_img)

        return {
            "pred_images": torch.stack(pred_images, dim=1),
            "pred_joints": torch.stack(pred_joints, dim=1),
            "pred_points": torch.stack(pred_points, dim=1),
            "enc_points":  torch.stack(enc_points, dim=1),
        }

In [ ]:
# モデルの動作確認
model = HSARNN(image_shape=(3, 64, 64), joint_dim=7, n_keypoints=5).to(device)

# パラメータ数
n_params = sum(p.numel() for p in model.parameters())
print(f"パラメータ数: {n_params:,}")

# ダミー入力
dummy_imgs = torch.randn(2, 10, 3, 64, 64, device=device)
dummy_jnts = torch.randn(2, 10, 7, device=device)

out = model(dummy_imgs, dummy_jnts)
for k, v in out.items():
    print(f"{k}: {v.shape}")

## 6. BPTTによる学習

### 損失関数

HSARNNでは以下の3つの損失を組み合わせます:

| 損失 | 内容 | 重み |
|---|---|---|
| 画像再構成損失 | 予測画像と実画像のMSE | `w_img` |
| 関節角度損失 | 予測関節角度と実関節角度のMSE | `w_jnt` |
| キーポイント損失 | 予測キーポイントとエンコーダ出力のMSE | `w_pt` (アニーリング) |

**キーポイント損失のアニーリング**: 学習初期はキーポイント損失の重みを高く設定し、
エンコーダの出力を安定させます。学習が進むにつれて重みを徐々に下げ、
RNNが自律的に予測できるようにします。

In [ ]:
class RobomimicDummyDataset(Dataset):
    """ダミーの Robomimic 風データセット"""
    def __init__(self, n_episodes=20, seq_len=30, image_shape=(3, 64, 64), joint_dim=7):
        self.data = []
        for _ in range(n_episodes):
            # ランダムな軌道を生成
            joints = torch.cumsum(torch.randn(seq_len, joint_dim) * 0.1, dim=0)
            images = torch.rand(seq_len, *image_shape) * 0.5
            # 画像に関節角度依存のパターンを埋め込む
            for t in range(seq_len):
                cx = int((torch.sigmoid(joints[t, 0]) * (image_shape[1] - 1)).item())
                cy = int((torch.sigmoid(joints[t, 1]) * (image_shape[2] - 1)).item())
                r = 3
                x0, x1 = max(0, cx - r), min(image_shape[1], cx + r)
                y0, y1 = max(0, cy - r), min(image_shape[2], cy + r)
                images[t, 0, x0:x1, y0:y1] = 1.0  # 赤いドット
            self.data.append((images, joints))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]  # (images, joints)


dataset = RobomimicDummyDataset(n_episodes=50, seq_len=20)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
print(f"データセットサイズ: {len(dataset)}, バッチ数: {len(dataloader)}")

In [ ]:
def train_hsarnn(model, dataloader, n_epochs=50, lr=1e-3,
                 w_img=1.0, w_jnt=1.0, w_pt_start=10.0, w_pt_end=0.1):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "loss_img": [], "loss_jnt": [], "loss_pt": []}

    for epoch in range(n_epochs):
        # キーポイント損失のアニーリング（線形減衰）
        progress = epoch / max(n_epochs - 1, 1)
        w_pt = w_pt_start + (w_pt_end - w_pt_start) * progress

        epoch_losses = {"loss": 0, "loss_img": 0, "loss_jnt": 0, "loss_pt": 0}
        n_batches = 0

        for images, joints in dataloader:
            images = images.to(device)
            joints = joints.to(device)

            out = model(images, joints)

            # 1ステップ先予測との比較（t → t+1）
            loss_img = F.mse_loss(out["pred_images"][:, :-1], images[:, 1:])
            loss_jnt = F.mse_loss(out["pred_joints"][:, :-1], joints[:, 1:])
            loss_pt  = F.mse_loss(out["pred_points"], out["enc_points"].detach())

            loss = w_img * loss_img + w_jnt * loss_jnt + w_pt * loss_pt

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_losses["loss"] += loss.item()
            epoch_losses["loss_img"] += loss_img.item()
            epoch_losses["loss_jnt"] += loss_jnt.item()
            epoch_losses["loss_pt"] += loss_pt.item()
            n_batches += 1

        for k in epoch_losses:
            epoch_losses[k] /= n_batches
            history[k].append(epoch_losses[k])

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}/{n_epochs} | "
                  f"Loss: {epoch_losses['loss']:.4f} | "
                  f"Img: {epoch_losses['loss_img']:.4f} | "
                  f"Jnt: {epoch_losses['loss_jnt']:.4f} | "
                  f"Pt: {epoch_losses['loss_pt']:.4f} | "
                  f"w_pt: {w_pt:.2f}")

    return history

In [ ]:
model = HSARNN(image_shape=(3, 64, 64), joint_dim=7, n_keypoints=5,
               rnn_dim=50, union_dim=20).to(device)

history = train_hsarnn(model, dataloader, n_epochs=50, lr=1e-3)

## 7. 学習と可視化

学習曲線と予測結果を可視化します:
- **入力画像** vs **予測画像**
- **予測注意点（キーポイント）** の位置
- **関節角度** の予測精度

In [ ]:
# 学習曲線の可視化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["loss_img"], label="Image Loss")
axes[0].set_title("画像再構成損失")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["loss_jnt"], label="Joint Loss", color="orange")
axes[1].set_title("関節角度損失")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(history["loss_pt"], label="Point Loss", color="green")
axes[2].set_title("キーポイント損失")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 予測結果の可視化
model.eval()
images_vis, joints_vis = dataset[0]
images_vis = images_vis.unsqueeze(0).to(device)
joints_vis = joints_vis.unsqueeze(0).to(device)

with torch.no_grad():
    out = model(images_vis, joints_vis)

T_show = min(5, images_vis.shape[1] - 1)  # 表示するタイムステップ数
fig, axes = plt.subplots(4, T_show, figsize=(3 * T_show, 12))

for t in range(T_show):
    # 入力画像
    img_in = images_vis[0, t + 1].cpu().permute(1, 2, 0).clamp(0, 1).numpy()
    axes[0, t].imshow(img_in)
    axes[0, t].set_title(f"入力 t={t+1}")
    axes[0, t].axis("off")

    # 予測画像
    img_pred = out["pred_images"][0, t].cpu().permute(1, 2, 0).clamp(0, 1).numpy()
    axes[1, t].imshow(img_pred)
    axes[1, t].set_title(f"予測 t={t+1}")
    axes[1, t].axis("off")

    # 予測注意点（キーポイント）を入力画像上にプロット
    axes[2, t].imshow(img_in)
    pts = out["pred_points"][0, t].cpu().view(-1, 2).numpy()
    # キーポイント座標を [-1,1] -> ピクセル座標に変換
    px = (pts[:, 1] + 1) / 2 * 64
    py = (pts[:, 0] + 1) / 2 * 64
    axes[2, t].scatter(px, py, c="red", s=50, marker="x", linewidths=2)
    axes[2, t].set_title(f"注意点 t={t+1}")
    axes[2, t].axis("off")

    # 関節角度の比較
    gt = joints_vis[0, t + 1].cpu().numpy()
    pred = out["pred_joints"][0, t].cpu().numpy()
    x_idx = np.arange(len(gt))
    axes[3, t].bar(x_idx - 0.15, gt, 0.3, label="真値", alpha=0.7)
    axes[3, t].bar(x_idx + 0.15, pred, 0.3, label="予測", alpha=0.7)
    axes[3, t].set_title(f"関節角度 t={t+1}")
    if t == 0:
        axes[3, t].legend(fontsize=8)

axes[0, 0].set_ylabel("入力画像", fontsize=12)
axes[1, 0].set_ylabel("予測画像", fontsize=12)
axes[2, 0].set_ylabel("予測注意点", fontsize=12)
axes[3, 0].set_ylabel("関節角度", fontsize=12)

plt.tight_layout()
plt.show()

## 8. 演習問題

### 演習1: HierachicalRNNCellのforward実装

以下のテンプレートに従って、`HierachicalRNNCell` の `forward` メソッドを実装してください。

**ヒント:**
1. 前時刻の `io_rnn1` と `io_rnn2` の隠れ状態を結合して `union_rnn` に入力
2. `union_out` で出力を分割し、各 io_rnn にフィードバック
3. 各 io_rnn を更新

In [ ]:
class MyHierachicalRNNCell(nn.Module):
    def __init__(self, input_dim1, input_dim2, rnn_dim=50, union_dim=20):
        super().__init__()
        self.rnn_dim = rnn_dim
        self.union_dim = union_dim
        self.io_rnn1 = nn.LSTMCell(input_dim1, rnn_dim)
        self.io_rnn2 = nn.LSTMCell(input_dim2, rnn_dim)
        self.union_rnn = nn.LSTMCell(rnn_dim * 2, union_dim)
        self.union_out = nn.Linear(union_dim, rnn_dim * 2)

    def get_initial_states(self, x):
        B, dev = x.shape[0], x.device
        z = lambda d: torch.zeros(B, d, device=dev)
        return [z(self.rnn_dim),z(self.rnn_dim)],[z(self.rnn_dim),z(self.rnn_dim)],[z(self.union_dim),z(self.union_dim)]

    def forward(self, xv, xpt, states=None):
        if states is None:
            states = self.get_initial_states(xv)
        prev_rnn1, prev_rnn2, prev_union = states

        # ここにコードを書いてください
        # 1. union_rnn に prev_rnn1[0] と prev_rnn2[0] を結合して入力
        # 2. union_out で分割
        # 3. io_rnn1, io_rnn2 を更新
        pass

# テスト
cell_test = MyHierachicalRNNCell(7, 10).to(device)
xv_t = torch.randn(4, 7, device=device)
xpt_t = torch.randn(4, 10, device=device)
r1, r2, u = cell_test(xv_t, xpt_t)
print(f"io_rnn1 h: {r1[0].shape}, io_rnn2 h: {r2[0].shape}, union h: {u[0].shape}")

<details>
<summary><b>回答例を表示</b></summary>

```python
def forward(self, xv, xpt, states=None):
    if states is None:
        states = self.get_initial_states(xv)
    prev_rnn1, prev_rnn2, prev_union = states

    # 1. Union RNN
    inputs_u = torch.cat((prev_rnn1[0], prev_rnn2[0]), dim=-1)
    new_union = self.union_rnn(inputs_u, prev_union)

    # 2. Union出力を分割
    _uh = self.union_out(new_union[0])
    r1s, r2s = torch.split(_uh, self.rnn_dim, dim=-1)

    # 3. 各RNNを更新
    new_rnn1 = self.io_rnn1(xv, [r1s, prev_rnn1[1]])
    new_rnn2 = self.io_rnn2(xpt, [r2s, prev_rnn2[1]])

    return new_rnn1, new_rnn2, new_union
```

</details>

### 演習2: HSARNNモデル全体の組み立てとダミーデータ動作確認

以下のテンプレートを完成させて、HSARNNモデルを構築してください。
`forward` メソッドでは、各タイムステップで画像エンコード→階層RNN→デコードを行います。

**確認ポイント:**
- 出力の各テンソルの形状が正しいこと
- バッチサイズ2、系列長5で動作すること

In [ ]:
class MyHSARNN(nn.Module):
    def __init__(self, image_shape=(3, 64, 64), joint_dim=7,
                 n_keypoints=5, rnn_dim=50, union_dim=20):
        super().__init__()
        C, H, W = image_shape
        self.n_keypoints = n_keypoints

        # ここにコードを書いてください
        # encoder, spatial_softmax, rnn_cell, decoders を定義
        pass

    def forward(self, images, joints):
        B, T = images.shape[:2]
        states = None
        pred_images, pred_joints, pred_points, enc_points = [], [], [], []

        for t in range(T):
            # ここにコードを書いてください
            # 1. 画像エンコード
            # 2. 階層RNN
            # 3. デコード
            pass

        return {
            "pred_images": torch.stack(pred_images, dim=1),
            "pred_joints": torch.stack(pred_joints, dim=1),
            "pred_points": torch.stack(pred_points, dim=1),
            "enc_points":  torch.stack(enc_points, dim=1),
        }

# テスト
my_model = MyHSARNN().to(device)
test_imgs = torch.randn(2, 5, 3, 64, 64, device=device)
test_jnts = torch.randn(2, 5, 7, device=device)
test_out = my_model(test_imgs, test_jnts)
for k, v in test_out.items():
    print(f"{k}: {v.shape}")

<details>
<summary><b>回答例を表示</b></summary>

```python
# 上記セクション5のHSARNNクラスを参照してください。
# ポイント:
# - encoder: Conv2d x3 で (B,C,H,W) -> (B,n_kp,H/4,W/4)
# - SpatialSoftmax で特徴マップ -> キーポイント座標
# - HierachicalRNNCell(joint_dim, n_kp*2, rnn_dim, union_dim)
# - joint_decoder: Linear(rnn_dim, joint_dim)
# - point_decoder: Linear(rnn_dim, n_kp*2)
# - image_decoder: InverseSpatialSoftmax + ConvTranspose で再構成
```

</details>

### 演習3: SARNNとHSARNNの比較

以下の簡易SARNNモデルとHSARNNモデルを同じダミーデータで学習し、
関節角度の予測精度（MSE）を比較してください。

**比較ポイント:**
- 学習曲線の収束速度
- 最終的な関節角度の予測誤差
- パラメータ数の違い

In [ ]:
# 簡易SARNNモデル（単一RNN）
class SimpleSARNN(nn.Module):
    def __init__(self, image_shape=(3, 64, 64), joint_dim=7,
                 n_keypoints=5, rnn_dim=50):
        super().__init__()
        C, H, W = image_shape
        self.n_keypoints = n_keypoints
        self.encoder = nn.Sequential(
            nn.Conv2d(C, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, n_keypoints, 3, padding=1),
        )
        self.spatial_softmax = SpatialSoftmax()
        self.rnn = nn.LSTMCell(joint_dim + n_keypoints * 2, rnn_dim)
        self.joint_decoder = nn.Linear(rnn_dim, joint_dim)
        self.point_decoder = nn.Linear(rnn_dim, n_keypoints * 2)
        self.rnn_dim = rnn_dim

    def forward(self, images, joints):
        B, T = images.shape[:2]
        h = torch.zeros(B, self.rnn_dim, device=images.device)
        c = torch.zeros(B, self.rnn_dim, device=images.device)
        pred_joints, pred_points = [], []
        for t in range(T):
            feat = self.encoder(images[:, t])
            pts = self.spatial_softmax(feat)
            inp = torch.cat([joints[:, t], pts], dim=-1)
            h, c = self.rnn(inp, (h, c))
            pred_joints.append(self.joint_decoder(h))
            pred_points.append(self.point_decoder(h))
        return {
            "pred_joints": torch.stack(pred_joints, dim=1),
            "pred_points": torch.stack(pred_points, dim=1),
        }

# ここにコードを書いてください
# 1. SimpleSARNN と HSARNN をそれぞれ初期化
# 2. 同じデータで学習（関節角度損失のみで比較可能）
# 3. 学習曲線を比較プロット

<details>
<summary><b>回答例を表示</b></summary>

```python
sarnn_model = SimpleSARNN().to(device)
hsarnn_model = HSARNN().to(device)

print(f"SARNN params:  {sum(p.numel() for p in sarnn_model.parameters()):,}")
print(f"HSARNN params: {sum(p.numel() for p in hsarnn_model.parameters()):,}")

def train_simple(model, dataloader, n_epochs=50, lr=1e-3, is_hsarnn=False):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    jnt_losses = []
    for epoch in range(n_epochs):
        total = 0; n = 0
        for images, joints in dataloader:
            images, joints = images.to(device), joints.to(device)
            out = model(images, joints)
            loss = F.mse_loss(out["pred_joints"][:, :-1], joints[:, 1:])
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item(); n += 1
        jnt_losses.append(total / n)
    return jnt_losses

sarnn_losses = train_simple(sarnn_model, dataloader)
hsarnn_losses = train_simple(hsarnn_model, dataloader, is_hsarnn=True)

plt.figure(figsize=(8, 5))
plt.plot(sarnn_losses, label="SARNN")
plt.plot(hsarnn_losses, label="HSARNN")
plt.xlabel("Epoch"); plt.ylabel("Joint MSE")
plt.title("SARNNとHSARNNの比較")
plt.legend(); plt.show()
```

</details>

## まとめ

このノートブックでは以下を学びました:

1. **SARNNの限界**: 単一RNNで異なるモダリティを処理する際の課題
2. **HierachicalRNNCell**: 関節角度とキーポイントを分離処理し、Union RNNで統合する階層構造
3. **HSARNNモデル**: SpatialSoftmax + 階層型RNN + デコーダの統合アーキテクチャ
4. **BPTTとアニーリング**: 画像・関節・キーポイントの3損失と、キーポイント損失の重み減衰
5. **可視化**: 入力画像、予測画像、注意点、関節角度の4種類の出力を確認

### 次のステップ

- 実際のロボットデータセット（Robomimic など）でHSARNNを学習
- キーポイント数やRNN次元のハイパーパラメータ調整
- 3つ以上のモダリティへの拡張（触覚、力覚など）